In [ ]:
### 주성분 분석(PCA)

## 외부 데이터 저장 및 넘파이 형식으로 로드
!wget https://bit.ly/fruits_300_data -O fruits_300.npy

import numpy as np
fruits = np.load('fruits_300.npy')
fruits_2d = fruits.reshape(-1, 100*100)



## PCA 클래스 객체 생성 및 활용
from sklearn.decomposition import PCA
pca = PCA(n_components = 50) # 주성분 개수 지정(50개)
pca.fit(fruits_2d)
print(pca.components_.shape) # components => PCA 객체가 찾은 주성분 데이터 저장

# 06_2에서 사용한 draw_fruits 함수 재사용

import matplotlib.pyplot as plt

def draw_fruits(arr, ratio = 1): # 입력 인수 => 배열
  n = len(arr)

  rows = int(np.ceil(n/10))
  cols = n if rows < 2 else 10

  fig, axs = plt.subplots(rows, cols, figsize = (cols*ratio, rows*ratio), squeeze = False)

  for i in range(rows):
    for j in range(cols):
      if i*10 + j < n:
        axs[i, j].imshow(arr[i*10 + j], cmap = 'gray_r')
        axs[i, j].axis('off')
  plt.show()

draw_fruits(pca.components_.reshape(-1, 100, 100))

# PCA를 활용하여 원본 데이터 차원 축소
print("차원 축소 전 원본 데이터 크기: ",fruits_2d.shape)
# 축소된 차원(특성 개수) = 주성분 개수(즉 주성분 개수만큼 차원 단축 가능)
fruits_pca = pca.transform(fruits_2d)
print("차원 축소 후 원본 데이터 크기: ", fruits_pca.shape)



## 원본 데이터 재구성

# inverse_transform => 원본 데이터 재구성 메서드
fruits_inverse = pca.inverse_transform(fruits_pca)

# 재구성한 원본 데이터 특성을 2차원으로(행과 열로) 배치 후 시각화
fruits_reconstruct = fruits_inverse.reshape(-1, 100, 100)
for start in [0, 100, 200]: # 샘플 100개 단위로 시각화
  draw_fruits(fruits_reconstruct[start:start+100])
  print("\n")



## 설명된 분산
print(np.sum(pca.explained_variance_ratio_)) # 설명된 분산 비율 총합

plt.plot(pca.explained_variance_ratio_)
plt.show()



## PCA 기법을 지도 학습 알고리즘(로지스틱)에 적용 => PCA 성능 확인

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

# 지도 학습 => 타깃이 필요하다 => "이미 정답을 아는 훈련 데이터"로부터 타깃 형성
target = np.array([0] * 100 + [1] * 100 + [2] * 100)

from sklearn.model_selection import cross_validate

scores = cross_validate(lr, fruits_2d, target)
print("PCA 이전 지도 학습 알고리즘의 교차 점수 평균: ", np.mean(scores['test_score']))
print("PCA 이전 지도 학습 알고리즘의 교차 검증 소요 시간: ",np.mean(scores['fit_time']))

# PCA 기법을 거쳐 차원을 단축한 훈련 데이터 사용
scores = cross_validate(lr, fruits_pca, target)
print("PCA 이후 지도 학습 알고리즘의 교차 점수 평균: ", np.mean(scores['test_score']))
print("PCA 이후 지도 학습 알고리즘의 교차 검증 소요 시간: ",np.mean(scores['fit_time'])) # 정확도 유지 & 시간 약 50배 단축

# PCA 객체 생성 시 주성분 개수 대신 설명된 분산 비율도 지정 가능(n_components)
pca = PCA(n_components = 0.5) # n_components에 0~1 사이 입력 => 설명된 분산 비율
pca.fit(fruits_2d)
print(pca.n_components_)

fruits_pca = pca.transform(fruits_2d)
print("차원 축소 후 원본 데이터 크기: ", fruits_pca.shape)

scores = cross_validate(lr, fruits_pca, target)
print("PCA 이후 지도 학습 알고리즘의 교차 점수 평균: ", np.mean(scores['test_score']))
print("PCA 이후 지도 학습 알고리즘의 교차 검증 소요 시간: ",np.mean(scores['fit_time']))



## PCA를 적용하여 K-평균 알고리즘 실행

from sklearn.cluster import KMeans
km = KMeans(n_clusters = 3, random_state = 42)
km.fit(fruits_pca)
print(np.unique(km.labels_, return_counts = True))

for label in range(0, 3):
  draw_fruits(fruits[km.labels_ == label])
  print("\n")

# PCA의 또 다른 장점 - 비교적 쉽게 시각화 가능

for label in range(0, 3): # 각 클러스터에 속하는 샘플을 산점도 표현
  data = fruits_pca[km.labels_ == label]
  plt.scatter(data[:, 0], data[:, 1])

plt.legend(['apple', 'banana', 'pineapple'])
plt.show()
